In [1]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pypdf import PdfReader
reader = PdfReader("Data_pdfs/RAG.pdf")
rag = []
for index, page in enumerate(reader.pages):
    rag.append(page.extract_text())
reader0 = PdfReader("Data_pdfs/Attention_Is_All_You_Need.pdf")
aiayn = []
for index, page in enumerate(reader0.pages):
    aiayn.append(page.extract_text())
reader1 = PdfReader("Data_pdfs/LoRA.pdf")
lora = []
for index, page in enumerate(reader1.pages):
    lora.append(page.extract_text())
reader2 = PdfReader("Data_pdfs/QLoRA.pdf")

qlora = []
for index, page in enumerate(reader2.pages):
    qlora.append(page.extract_text())
reader3 = PdfReader("Data_pdfs/BERT.pdf")
bert = []
for index, page in enumerate(reader3.pages):
    bert.append(page.extract_text())

In [7]:
docs = {"RAG": rag, "AIAYN": aiayn, "LoRA": lora, "QLoRA": qlora, "BERT": bert}

In [17]:
len(docs)

5

In [8]:
import re

def clean(text):
    text = re.sub(r"-\n", "", text)    # join hyphenated line breaks: "exam-\nple" -> "example"
    text = re.sub(r"\n", " ", text)    # remaining newlines -> spaces
    text = re.sub(r"\s+", " ", text)   # collapse repeated whitespace
    return text.strip()

In [11]:
%pip install langchain-text-splitters

   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ------------------ --------------------- 262.1/580.5 kB ? eta -:--:--
   ------------------------------------ --- 524.3/580.5 kB 1.1 MB/s eta 0:00:01
   ---------------------------------------- 580.5/580.5 kB 1.1 MB/s  0:00:00
   ---------------------------------------- 0.0/940.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/940.8 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/940.8 kB 1.5 MB/s eta 0:00:01
   --------------------------------- ------ 786.4/940.8 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 940.8/940.8 kB 1.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   --------- ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
chunks = []
chunk_id = 0
for pdf_name, pages in docs.items():
    for page_num, text in enumerate(pages, start=1):
        text = clean(text)
        if not text: continue
        pieces = splitter.split_text(text)
        for piece in pieces:
            if len(piece) < 100: continue
            chunks.append({"pdf_name": pdf_name, "page": page_num,
                           "chunk_id": chunk_id, "text": piece })
            chunk_id += 1

In [19]:
len(chunks)

300

In [20]:
%pip install sentence_transformers

   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ------------- -------------------------- 262.1/760.7 kB ? eta -:--:--
   ---------------------------------------- 760.7/760.7 kB 2.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [11]:
from sentence_transformers import SentenceTransformer

W1110 00:48:43.129000 17356 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c['text'] for c in chunks]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
embeddings = model.encode(texts , normalize_embeddings = True  , batch_size = 64 , show_progress_bar = True)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [31]:
pip install faiss-cpu

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.3 MB 8.4 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/16.3 MB 13.0 MB/s eta 0:00:01
   ----------------- ---------------------- 7.1/16.3 MB 12.8 MB/s eta 0:00:01
   ------------------------ --------------- 10.0/16.3 MB 13.8 MB/s eta 0:00:01
   -------------------------------- ------- 13.4/16.3 MB 14.0 MB/s eta 0:00:01
   ---------------------------------------  16.3/16.3 MB 14.0 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 13.1 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [14]:
import faiss
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

In [18]:
def retrieve(query , k = 5):
    q_embed = model.encode([query] , normalize_embeddings = True)
    scores , ids = index.search(q_embed , k)
    retrieved_docs = [[chunks[idx], score] for idx, score in zip(ids[0], scores[0]) if idx != -1]
    return retrieved_docs

In [49]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="auto")
def generate(query , k = 5 , max_new_tokens = 256):
    docs = retrieve(query , k)
    context = "\n\n".join( f"[{c['pdf_name']}, p.{c['page']}]\n{c['text']}" for c, _ in docs)
    messages = [{"role":"system" , "content":"Answer using only the context. Cite sources like [1]. If the answer isn't in the context, say you don't know."},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}]
    prompt = tok.apply_chat_template(messages , tokenize=False , add_generation_prompt = True)
    inputs = tok(prompt , return_tensors = "pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs.input_ids.shape[1]:] , skip_special_token = True).replace("<|im_end|>", "").strip() , docs
                
    

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [28]:
answer, docs = generate("what is Q in QLoRA")
print(answer)

QLoRA stands for "Quantized Low-Rank Adaptation."


In [29]:
llm.device

device(type='cuda', index=0)

In [30]:
chunks

[{'pdf_name': 'RAG',
  'page': 1,
  'chunk_id': 0,
  'text': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks Patrick Lewis†‡, Ethan Perez⋆, Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†, Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela† †Facebook AI Research;‡University College London;⋆New York University; plewis@fb.com Abstract Large pre-trained language models have been shown to store factual knowledge in their parameters, and achieve state-of-the-art results when ﬁne-tuned on downstream NLP tasks. However, their ability to access and precisely manipulate knowledge is still limited, and hence on knowledge-intensive tasks, their performance lags behind task-speciﬁc architectures. Additionally, providing provenance for their decisions and updating their world knowledge remain open research problems. Pretrained models with a differentiable access mechanism to explicit non-parametric memory have so 

In [31]:
eval_set = [
 {"q":"How many attention heads does the base Transformer use, and what is d_k per head?","pdf_name":"AIAYN","pages":[5]},
 {"q":"What dropout rate does the base Transformer use?","pdf_name":"AIAYN","pages":[8]},
 {"q":"Why are dot products scaled by 1/sqrt(d_k)?","pdf_name":"AIAYN","pages":[4]},
 {"q":"How many warmup steps does the learning rate schedule use?","pdf_name":"AIAYN","pages":[7]},

 {"q":"What fraction of tokens is masked in BERT's masked LM?","pdf_name":"BERT","pages":[4,12,13]},
 {"q":"Of the tokens selected for masking, how are they replaced?","pdf_name":"BERT","pages":[4,12]},
 {"q":"What are the layers, hidden size, heads and parameters of BERT_BASE?","pdf_name":"BERT","pages":[3]},
 {"q":"Which corpora was BERT pre-trained on?","pdf_name":"BERT","pages":[5,14]},

 {"q":"Which attention weight matrices does LoRA adapt in most experiments?","pdf_name":"LoRA","pages":[5,6]},
 {"q":"How are LoRA matrices A and B initialized?","pdf_name":"LoRA","pages":[1,4]},
 {"q":"By how much does LoRA reduce trainable parameters and GPU memory on GPT-3 175B?","pdf_name":"LoRA","pages":[1,5]},
 {"q":"Why does LoRA add no inference latency?","pdf_name":"LoRA","pages":[2,4]},

 {"q":"What three innovations does QLoRA introduce?","pdf_name":"QLoRA","pages":[1,2]},
 {"q":"What does Double Quantization quantize?","pdf_name":"QLoRA","pages":[1,2,5]},
 {"q":"What GPU memory is enough to finetune a 65B model with QLoRA?","pdf_name":"QLoRA","pages":[1,2]},
 {"q":"What percentage of ChatGPT's Vicuna-benchmark performance does Guanaco 65B reach?","pdf_name":"QLoRA","pages":[1,2,9,10]},

 {"q":"What are the two RAG formulations?","pdf_name":"RAG","pages":[3]},
 {"q":"Which retriever does RAG use?","pdf_name":"RAG","pages":[2,3]},
 {"q":"Which generator model does RAG use?","pdf_name":"RAG","pages":[2,3]},
 {"q":"How is Wikipedia split for the RAG index, and how many documents result?","pdf_name":"RAG","pages":[4,19]},
]

In [47]:

recalls = {1: 0, 2: 0, 3: 0, 4: 0, 5: 0}
for q in eval_set:
    d = retrieve(q['q'])
    step = 1
    for u in d:
        if(u[0]['pdf_name'] == q['pdf_name']):
            if(u[0]['page'] in q['pages']):
                for k in range(step, 6):
                  recalls[k] += 1
                break

        step+= 1
print(recalls)

{1: 12, 2: 15, 3: 17, 4: 18, 5: 19}


True


In [52]:
print(generate("How is Wikipedia split for the RAG index, and how many documents result?")[0])

Wikipedia is split into multiple smaller files, resulting in approximately 100 GB of CPU memory being required after indexing.


In [2]:
!nvidia-smi

Tue Nov 10 01:44:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.19                 Driver Version: 592.19         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   37C    P0             25W /  115W |       0MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch
import gc

# 1. Move the model to CPU memory (RAM) to clear it from the GPU
if 'model' in globals():
    model.cpu() 
    del model

# 2. Clear out any other lingering variables
gc.collect()

# 3. Release the cached GPU memory
torch.cuda.empty_cache()

In [5]:
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")

0.0 GB allocated


In [6]:
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

0.0 GB reserved
